# 随手练习 手撕 笔记本

# 手撕MHA

In [ ]:
import torch.nn as nn
import torch.functional as F

import torch
import math

In [ ]:
def scaled_dot_product_attention(q,k,v,mask=None):
    scale = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(scale)

    if mask is not None:
        scores = scores.masked_fill(mask==0, torch.float("-inf")) # 这里也可以用 -1e9
    attention = F.softmax(scores,dim = -1)

    output = torch.matmul(attention, v)
    return output, attention 


class MHA(nn.Module):
    def __init__(self, d_model,num_head):
        super(MHA, self).__init__()

        self.w_q = nn.Linear(d_model,d_model)
        self.w_k = nn.Linear(d_model,d_model)
        self.w_v = nn.Linear(d_model,d_model)
        self.h = num_head
        self.d_model = d_model
        assert d_model % num_head ==0 , "头的数量必须可以被d_model整除"
        self.fc_out = nn.Linear(d_model,d_model)

    def forward(self, q,k,v,mask=None):
        """
        假设输入的q k v 是没有经过处理的单头

        """
        batch_size = q.size(0)
        seq_len_q = q.size(1)
        seq_len_k = k.size(1)
        Q = self.w_q(q).view(batch_size, seq_len_q, self.h, -1).transpose(1,2)
        K = self.w_k(k).view(batch_size, seq_len_k, self.h, -1).transpose(1,2)
        V = self.w_v(v).view(batch_size, seq_len_k, self.h, -1).transpose(1,2)

        output, atten = scaled_dot_product_attention(Q,K,V, mask = mask)

        out = output.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)

        out = self.fc_out(out)

        return out 

# 手撕rope

In [ ]:
class RoPE(nn.Module):
    def __init_(self,seq_len,dim ):
        super(RoPE, self).__init__()
        # 初始化 position * 频率
        freq_term = 1.0/10000 **(torch.arange(0, dim, step=2).float()/dim)  # dim//2
        position = torch.arange(seq_len).float()# seq_len

        freqs = torch.einsum("i,j ->ij", position, freq_term)
        embed = torch.cat((freqs, freqs), dim =-1) # (seq_len, dim)

        self.register_buffer("cos_cached", embed.cos(), persistent=False)
        self.register_buffer("sin_cached", embed.sin(), persistent=False)

    def forward(self, q,k, position_ids: torch.Tensor = None):
        cos = self.cos_cached[position_ids].unsqueeze(0).unsqueeze(2)
        sin = self.sin_cached[position_ids].unsqueeze(0).unsqueeze(2)
        # position_ids应该是是 (seq_len,)
        embed_q = cos*q + sin * self.rotate_half(q) 
        embed_k = cos*k + sin * self.rotate_half(k) 

        return embed_q, embed_k
    
    @staticmethod
    def rotate_half(x):
        x1 = x[..., :x.shape[-1]//2]
        x2 = x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1), dim = -1)